In [1]:
!pip install -q transformers peft bitsandbytes accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00


In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel

In [3]:
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [4]:
model = PeftModel.from_pretrained(model, "/content/adapters")

In [5]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("/content/merged_model")
tokenizer.save_pretrained("/content/merged_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/merged_model/tokenizer_config.json',
 '/content/merged_model/chat_template.jinja',
 '/content/merged_model/tokenizer.json')

In [6]:
!du -sh /content/merged_model

2.1G	/content/merged_model


In [7]:
prompt = """### Instruction:
Answer the coding question clearly

### Input:
What is a loop in Python?

### Response:
"""

pipe_fp16 = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device=0
)

start = time.time()
result = pipe_fp16(prompt, max_new_tokens=80)
end = time.time()

print(result[0]["generated_text"])
print("FP16 Time:", round(end - start, 2), "seconds")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Answer the coding question clearly

### Input:
What is a loop in Python?

### Response:
A loop is used to execute code repeatedly.
FP16 Time: 1.79 seconds


In [8]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-int8/tokenizer_config.json',
 '/content/quantized/model-int8/chat_template.jinja',
 '/content/quantized/model-int8/tokenizer.json')

In [10]:
!du -sh /content/quantized/model-int8

1.2G	/content/quantized/model-int8


In [11]:
pipe_int8 = pipeline(
    "text-generation",
    model=model_int8,
    tokenizer=tokenizer,
    device=0
)

start = time.time()
result = pipe_int8(prompt, max_new_tokens=80)
end = time.time()

print(result[0]["generated_text"])
print("INT8 Time:", round(end - start, 2), "seconds")

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Answer the coding question clearly

### Input:
What is a loop in Python?

### Response:
A loop is a repeated execution of code. This concept is widely used in programming.
INT8 Time: 2.69 seconds


In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-int4/tokenizer_config.json',
 '/content/quantized/model-int4/chat_template.jinja',
 '/content/quantized/model-int4/tokenizer.json')

In [14]:
!du -sh /content/quantized/model-int4

774M	/content/quantized/model-int4


In [15]:
pipe_int4 = pipeline(
    "text-generation",
    model=model_int4,
    tokenizer=tokenizer,
    device=0
)

start = time.time()
result = pipe_int4(prompt, max_new_tokens=80)
end = time.time()

print(result[0]["generated_text"])
print("INT4 Time:", round(end - start, 2), "seconds")

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Answer the coding question clearly

### Input:
What is a loop in Python?

### Response:
Loops are used to execute code repeatedly. This concept is widely used in programming.
INT4 Time: 1.13 seconds


In [16]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release

Cloning into 'llama.cpp'...
remote: Enumerating objects: 87674, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 87674 (delta 234), reused 131 (delta 123), pack-reused 87335 (from 3)
Receiving objects: 100% (87674/87674), 364.46 MiB | 32.10 MiB/s, done.
Resolving deltas: 100% (63387/63387), done.
Updating files: 100% (2576/2576), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "

In [17]:
!python convert_hf_to_gguf.py /content/merged_model --outfile /content/quantized/model.gguf --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.we

In [18]:
!./build/bin/llama-quantize /content/quantized/model.gguf /content/quantized/model-q4_0.gguf q4_0

main: build = 8735 (4ef9301e4)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/quantized/model.gguf' to '/content/quantized/model-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from /content/quantized/model.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048


In [19]:
!rm /content/quantized/model.gguf

In [20]:
!du -sh /content/quantized/model-q4_0.gguf

608M	/content/quantized/model-q4_0.gguf


In [21]:
start = time.time()

!./build/bin/llama-cli -m /content/quantized/model-q4_0.gguf -p "What is a loop in Python?" -n 80

end = time.time()

print("GGUF Time:", round(end - start, 2), "seconds")e



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8735-4ef9301e4
model      : model-q4_0.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern


> What is a loop in Python?

|-\|/-\|/- A loop is a repeated execution of code. In Python, a loop is usually used to iterate over a range (e.g. 1 to 10). Here's how to create a loop:

1. Identify the loop variable: in this example, we'll use 'i'
2. Declare the loop: loop through range(1, 1

[ Prompt: 27.6 t/s | Generation: 10.4

In [22]:
!du -sh /content/merged_model
!du -sh /content/quantized/model-int8
!du -sh /content/quantized/model-int4
!du -sh /content/quantized/model-q4_0.gguf

2.1G	/content/merged_model
1.2G	/content/quantized/model-int8
774M	/content/quantized/model-int4
608M	/content/quantized/model-q4_0.gguf


## Quantization Comparison

| Format | Size | Inference Time | Quality |
|--------|------|---------------|---------|
| FP16 | 2.1 GB | 1.79 sec | Very Good |
| INT8 | 1.2 GB | 2.69 sec | Very Good |
| INT4 | 774 MB | 1.14 sec | Very Good |
| GGUF q4_0 | 608 MB | 10.4 tok/s | Best (Detailed Response) |

### Observations

- FP16, INT8 and INT4 generated accurate and meaningful responses with minimal quality difference.
- INT8 reduced model size significantly while maintaining output quality.
- INT4 provided strong compression and the fastest GPU inference in this setup.
- GGUF q4_0 was the most memory-efficient format and generated the most detailed response in testing.
- Initial GGUF load time was higher because llama.cpp required model loading during the first run.
- Overall, quantization reduced memory usage while preserving useful model performance.

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
import shutil
shutil.move('/content/quantized/','/content/drive/MyDrive')


'/content/drive/MyDrive/quantized'